# Mohammed Baseline Models — World Cup Mean Scoring Margin (filled in + extended to 2026)

This is a working copy of `Mohammed_Baseline_Validation/mohammed_baseline_models_reconstructed.ipynb`, made so we could actually run the template (the original was a reusable *template* with its data-loading cells commented out and never executed against real files -- it only carried the previously reported numbers as a static reference table) and then extend it to test against the real, now-completed 2026 World Cup, without touching the original submitted midway-report notebook.

Primary outcome:

$$Y_{it}=\frac{GF_{it}-GA_{it}}{G_{it}}$$

**Original validation design:** train on 2018 World Cup team summaries, test on 2022 World Cup team summaries.
**Extension added here:** after confirming the reconstruction reproduces the originally reported numbers, apply the same 2018-fitted models to the real 2026 World Cup results as a second, independent test set.

Models compared:

- Historical mean baseline
- Linear regression using FIFA ordinal rank
- Linear regression using FIFA ranking points
- Decision tree
- Random forest

Known reported results from the original run (2018 train -> 2022 test):

| Model | RMSE | MAE |
|---|---:|---:|
| Linear regression: FIFA rank | 0.9002 | 0.7212 |
| Decision tree | 0.9715 | 0.8280 |
| Random forest | 0.9767 | 0.7980 |
| Historical mean | 1.0224 | 0.8055 |
| Linear regression: FIFA points | 1.2481 | 1.0193 |

In [1]:
# Configuration
from pathlib import Path

# All data files live right next to this notebook (Mohammed_2026_Evaluation_v2/),
# unlike the original template which pointed at a generic "../data" folder.
DATA_DIR = Path(".")
FIGURE_DIR = DATA_DIR / "figures"
FIGURE_DIR.mkdir(parents=True, exist_ok=True)  # exist_ok=True: fine if this already exists from a prior run

WC2018_PATH = DATA_DIR / "wc_2018.csv"
WC2022_PATH = DATA_DIR / "wc_2022.csv"
WC2026_PATH = DATA_DIR / "wc_2026_model_input.csv"
FIFA_RANKINGS_PATH = DATA_DIR / "fifa_ranking-2024-06-20.csv"

# Pre-tournament FIFA ranking dates: the most recent ranking published before each
# World Cup kicked off. Using the pre-tournament (not mid- or post-tournament) ranking
# is what makes this a fair prediction setup -- the model only ever sees information
# that would have been available before a team's results were known.
RANK_DATE_2018 = "2018-06-07"
RANK_DATE_2022 = "2022-10-06"

print("Data files:", WC2018_PATH, WC2022_PATH, WC2026_PATH, FIFA_RANKINGS_PATH)

Data files: wc_2018.csv wc_2022.csv wc_2026_model_input.csv fifa_ranking-2024-06-20.csv


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

pd.set_option("display.max_columns", 100)  # so wide match-stat dataframes don't get truncated when displayed

## Helper functions

These are written to be adaptable because team datasets may use slightly different column names. If a function fails, inspect your dataframe columns and adjust the column mappings.

In [3]:
def clean_team_name(name):
    """Standardize team names for merging across datasets."""
    if pd.isna(name):
        return name
    name = str(name).strip().upper()
    # These are the specific mismatches we found between the wc_2018/wc_2022 files and
    # the FIFA rankings file's country_full column -- e.g. wc_2022.csv writes "UNITED STATES"
    # and "IRAN" in full, while the FIFA ranking file uses "USA" and "IR Iran". Uppercasing
    # both sides first, then applying this alias map, is enough to make every 2018/2022
    # team match its FIFA ranking row (verified below: zero unmatched teams either year).
    replacements = {
        "UNITED STATES": "USA",
        "UNITED STATES OF AMERICA": "USA",
        "IRAN": "IR IRAN",
        "KOREA REPUBLIC": "SOUTH KOREA",
        "KOREA, REPUBLIC OF": "SOUTH KOREA",
    }
    return replacements.get(name, name)


def compute_team_tournament_summary(team_match_df, team_col="team", gf_col="goals_for", ga_col="goals_against"):
    """Aggregate team-match rows to one row per team with mean scoring margin."""
    df = team_match_df.copy()
    df["team_clean"] = df[team_col].map(clean_team_name)
    out = (
        df.groupby("team_clean", as_index=False)
          .agg(
              goals_for=(gf_col, "sum"),       # total goals scored across every match played
              goals_against=(ga_col, "sum"),   # total goals conceded across every match played
              games_played=(team_col, "count") # counting rows per team = number of matches played
          )
    )
    # This is the project's target variable: (goals for - goals against) / games played.
    out["mean_scoring_margin"] = (out["goals_for"] - out["goals_against"]) / out["games_played"]
    return out


def match_rows_to_team_rows(match_df, team1_col, team2_col, goals1_col, goals2_col):
    """Convert one-row-per-match data into two team-match rows per match."""
    # wc_2022.csv has one row per MATCH (team1 vs. team2). We need one row per TEAM per
    # match instead (so compute_team_tournament_summary's groupby works the same way it
    # does for wc_2018.csv, which is already one row per team per match). We build that by
    # duplicating every match row twice: once from team1's perspective, once from team2's.
    a = match_df[[team1_col, team2_col, goals1_col, goals2_col]].copy()
    a.columns = ["team", "opponent", "goals_for", "goals_against"]
    b = match_df[[team2_col, team1_col, goals2_col, goals1_col]].copy()  # note goals columns swapped too
    b.columns = ["team", "opponent", "goals_for", "goals_against"]
    return pd.concat([a, b], ignore_index=True)


def get_pre_tournament_rankings(rankings_df, date_value, team_col="country_full", rank_col="rank", points_col="total_points", date_col="rank_date"):
    """Select FIFA rankings for a pre-tournament date and standardize columns."""
    r = rankings_df.copy()
    r[date_col] = pd.to_datetime(r[date_col])   # rank_date arrives as a string; compare as real dates
    date_value = pd.to_datetime(date_value)
    r = r.loc[r[date_col] == date_value].copy()  # keep only the one ranking release we asked for
    r["team_clean"] = r[team_col].map(clean_team_name)  # same aliasing used on the match data, so the merge key matches
    return r[["team_clean", rank_col, points_col]].rename(columns={rank_col: "fifa_rank", points_col: "fifa_points"})


def evaluate_regression(y_true, y_pred):
    # np.sqrt(mean_squared_error(...)) instead of the old squared=False kwarg: that
    # parameter was removed in newer scikit-learn (we're on 1.9.0 here), so this is the
    # version-independent way to get RMSE from mean_squared_error.
    return {
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "MAE": mean_absolute_error(y_true, y_pred),
    }

## Load and prepare data

You may need to adjust the column names in the cells below to match the team repository files.

In [4]:
wc2018_raw = pd.read_csv(WC2018_PATH)
wc2022_raw = pd.read_csv(WC2022_PATH)
rankings_raw = pd.read_csv(FIFA_RANKINGS_PATH)

# Printed once, up front, so it's obvious which raw column names the cells below assume --
# 2018 and 2022 use completely different schemas (one row per team-match vs. one row per match).
print("2018 columns:", wc2018_raw.columns.tolist())
print("2022 columns:", wc2022_raw.columns.tolist())
print("Rankings columns:", rankings_raw.columns.tolist())

2018 columns: ['Game', 'Group', 'Team', 'Opponent', 'Home/Away', 'Score', 'WDL', 'Pens?', 'Goals For', 'Goals Against', 'Pen Shootout For', 'Pen Shootout Against', 'Attempts', 'On-Target', 'Off-Target', 'Blocked', 'Woodwork', 'Corners', 'Offsides', 'Ball possession %', 'Pass Accuracy %', 'Passes', 'Passes Completed', 'Distance Covered km', 'Balls recovered', 'Tackles', 'Blocks', 'Clearances', 'Yellow cards', 'Red Cards', 'Second Yellow Card leading to Red Card', 'Fouls Committed']
2022 columns: ['team1', 'team2', 'possession team1', 'possession team2', 'possession in contest', 'number of goals team1', 'number of goals team2', 'date', 'hour', 'category', 'total attempts team1', 'total attempts team2', 'conceded team1', 'conceded team2', 'goal inside the penalty area team1', 'goal inside the penalty area team2', 'goal outside the penalty area team1', 'goal outside the penalty area team2', 'assists team1', 'assists team2', 'on target attempts team1', 'on target attempts team2', 'off targe

In [5]:
# 2018: already one row per team per match -- just rename to the column names
# compute_team_tournament_summary expects.
wc2018_team_rows = wc2018_raw.rename(columns={
    "Team": "team",
    "Goals For": "goals_for",
    "Goals Against": "goals_against",
})
summary_2018 = compute_team_tournament_summary(wc2018_team_rows, "team", "goals_for", "goals_against")

# 2022: one row per match (team1 vs team2) -- unpivot to one row per team per match first,
# using the goals-scored columns for each side of the match.
wc2022_team_rows = match_rows_to_team_rows(
    wc2022_raw,
    team1_col="team1",
    team2_col="team2",
    goals1_col="number of goals team1",
    goals2_col="number of goals team2",
)
summary_2022 = compute_team_tournament_summary(wc2022_team_rows)

# get_pre_tournament_rankings (defined above) applies the same clean_team_name aliasing
# to the rankings file's country names, so both sides of the join use matching keys.
rankings_2018 = get_pre_tournament_rankings(rankings_raw, RANK_DATE_2018)
rankings_2022 = get_pre_tournament_rankings(rankings_raw, RANK_DATE_2022)

# Inner join: only keep teams that appear in BOTH the tournament results and that date's
# FIFA rankings. If this dropped any of the 32 participating teams we'd see it in the
# "Unmatched" printout below -- it doesn't, so every team made it through cleanly.
train = summary_2018.merge(rankings_2018, on="team_clean", how="inner")
test = summary_2022.merge(rankings_2022, on="team_clean", how="inner")

print(f"train (2018): {len(train)} of {summary_2018['team_clean'].nunique()} teams matched to FIFA rankings")
print(f"test (2022): {len(test)} of {summary_2022['team_clean'].nunique()} teams matched to FIFA rankings")
missing_train = set(summary_2018['team_clean']) - set(train['team_clean'])
missing_test = set(summary_2022['team_clean']) - set(test['team_clean'])
print("Unmatched 2018 teams:", missing_train)
print("Unmatched 2022 teams:", missing_test)

train (2018): 32 of 32 teams matched to FIFA rankings
test (2022): 32 of 32 teams matched to FIFA rankings
Unmatched 2018 teams: set()
Unmatched 2022 teams: set()


## Model fitting and comparison

`run_baseline_models` (defined below, unchanged from the template) fits all five models on `train` (2018) and scores them on `test` (2022).

In [6]:
def run_baseline_models(train, test, random_state=42):
    y_train = train["mean_scoring_margin"].values
    y_test = test["mean_scoring_margin"].values

    results = []
    predictions = pd.DataFrame({
        "team_clean": test["team_clean"],
        "actual": y_test,
    })

    # Historical mean: the simplest possible baseline -- predict every 2022 team gets
    # exactly the average 2018 scoring margin, regardless of who they are. Any model
    # that can't beat this isn't learning anything useful from FIFA rank/points.
    hist_pred = np.repeat(y_train.mean(), len(test))
    predictions["pred_historical_mean"] = hist_pred
    results.append({"Model": "Historical mean", **evaluate_regression(y_test, hist_pred)})

    # Linear regression: FIFA rank (ordinal position, e.g. 1st, 2nd, 3rd...)
    rank_model = LinearRegression()
    rank_model.fit(train[["fifa_rank"]], y_train)
    rank_pred = rank_model.predict(test[["fifa_rank"]])
    predictions["pred_fifa_rank_lr"] = rank_pred
    results.append({"Model": "Linear regression: FIFA rank", **evaluate_regression(y_test, rank_pred)})

    # Linear regression: FIFA ranking points (continuous strength score, not just position)
    points_model = LinearRegression()
    points_model.fit(train[["fifa_points"]], y_train)
    points_pred = points_model.predict(test[["fifa_points"]])
    predictions["pred_fifa_points_lr"] = points_pred
    results.append({"Model": "Linear regression: FIFA points", **evaluate_regression(y_test, points_pred)})

    # Decision tree: given both rank and points, kept deliberately shallow (max_depth=3,
    # min_samples_leaf=3) because train is only 32 rows -- an unconstrained tree would
    # just memorize each of the 32 training teams individually and generalize terribly.
    tree = DecisionTreeRegressor(max_depth=3, min_samples_leaf=3, random_state=random_state)
    tree.fit(train[["fifa_rank", "fifa_points"]], y_train)
    tree_pred = tree.predict(test[["fifa_rank", "fifa_points"]])
    predictions["pred_decision_tree"] = tree_pred
    results.append({"Model": "Decision tree", **evaluate_regression(y_test, tree_pred)})

    # Random forest: 500 shallow trees (same max_depth/min_samples_leaf as above) averaged
    # together -- the averaging is meant to reduce the variance a single small tree has,
    # at the cost of losing the single tree's easy interpretability.
    forest = RandomForestRegressor(n_estimators=500, max_depth=3, min_samples_leaf=3, random_state=random_state)
    forest.fit(train[["fifa_rank", "fifa_points"]], y_train)
    forest_pred = forest.predict(test[["fifa_rank", "fifa_points"]])
    predictions["pred_random_forest"] = forest_pred
    results.append({"Model": "Random forest", **evaluate_regression(y_test, forest_pred)})

    results_df = pd.DataFrame(results).sort_values("RMSE")
    # Return the fitted model objects too (not just their scores) so they can be reused
    # later -- specifically, applied to the 2026 test set further down without refitting.
    models = {
        "rank_model": rank_model,
        "points_model": points_model,
        "tree": tree,
        "forest": forest,
    }
    return results_df, predictions, models

# results_df, predictions, models = run_baseline_models(train, test)
# display(results_df)
# print("Rank model intercept:", models["rank_model"].intercept_)
# print("Rank model coefficient:", models["rank_model"].coef_[0])

In [7]:
# Trains all five models on 2018 (train) and scores them on 2022 (test) in one call.
# fitted_models is kept around specifically so we can apply these same fitted models
# to the 2026 test set later, without retraining them on different data.
results_2022, predictions_2022, fitted_models = run_baseline_models(train, test)
results_2022

,Model,RMSE,MAE
1,Linear regression: FIFA rank,0.900249,0.721198
3,Decision tree,0.973625,0.808678
4,Random forest,0.999938,0.815185
0,Historical mean,1.022416,0.805497
2,Linear regression: FIFA points,1.248072,1.019342


## Plot actual vs predicted

This reproduces the type of figure included in the report.

In [8]:
def plot_actual_vs_predicted(predictions, pred_col="pred_fifa_rank_lr", output_path=FIGURE_DIR / "actual_vs_predicted_2022.png"):
    df = predictions.copy()
    df["abs_error"] = (df["actual"] - df[pred_col]).abs()  # used below to label the worst-predicted teams

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(df[pred_col], df["actual"], alpha=0.8)

    # Dashed diagonal = perfect prediction; points far from it are the biggest misses.
    min_val = min(df[pred_col].min(), df["actual"].min())
    max_val = max(df[pred_col].max(), df["actual"].max())
    ax.plot([min_val, max_val], [min_val, max_val], linestyle="--")

    # Label only the 5 worst-predicted teams by name, so the chart stays readable
    # instead of every one of the 32 points being annotated.
    for _, row in df.nlargest(5, "abs_error").iterrows():
        ax.annotate(row["team_clean"], (row[pred_col], row["actual"]), xytext=(5, 5), textcoords="offset points", fontsize=8)

    ax.set_xlabel("Predicted mean scoring margin")
    ax.set_ylabel("Actual mean scoring margin")
    ax.set_title("Actual vs Predicted 2022 Mean Scoring Margin")
    fig.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches="tight")
    return fig, ax

# plot_actual_vs_predicted(predictions)

## Sanity check: does our reconstruction reproduce the originally reported numbers?

The original notebook behind these numbers isn't in the repo (hence "reconstructed" -- this template was rebuilt from the midway report's written results). Before trusting this same pipeline to test 2026, we check whether re-running it from the raw 2018/2022 files gets RMSE/MAE reasonably close to what was reported. Small differences are expected (exact FIFA-rank-date handling, team-name matching, or which columns were binned may not be identical to whatever the original, unrecovered notebook did) -- large differences would mean this reconstruction isn't a faithful stand-in and its 2026 extension below shouldn't be trusted.

In [9]:
# Hard-coded reference values from the midway report -- the only surviving record of
# the original (lost) notebook's output. This is what results_2022 above gets compared
# against in the next cell, to check whether this reconstruction is trustworthy.
reported_results = pd.DataFrame({
    "Model": [
        "Linear regression: FIFA rank",
        "Decision tree",
        "Random forest",
        "Historical mean",
        "Linear regression: FIFA points",
    ],
    "RMSE": [0.9002, 0.9715, 0.9767, 1.0224, 1.2481],
    "MAE": [0.7212, 0.8280, 0.7980, 0.8055, 1.0193],
})
reported_results

,Model,RMSE,MAE
0,Linear regression: FIFA rank,0.9002,0.7212
1,Decision tree,0.9715,0.8280
2,Random forest,0.9767,0.7980
3,Historical mean,1.0224,0.8055
4,Linear regression: FIFA points,1.2481,1.0193


In [10]:
# Join our regenerated 2022 results to the originally reported ones by model name, so we
# can see, per model, exactly how far off (if at all) this reconstruction landed.
# Small diffs (a few thousandths) are expected version/randomness noise; large diffs would
# mean the data prep here doesn't match whatever the original, unrecovered notebook did.
comparison = results_2022.merge(reported_results, on="Model", suffixes=("_reconstructed", "_reported"))
comparison["RMSE_diff"] = comparison["RMSE_reconstructed"] - comparison["RMSE_reported"]
comparison["MAE_diff"] = comparison["MAE_reconstructed"] - comparison["MAE_reported"]
comparison.sort_values("Model")

,Model,RMSE_reconstructed,MAE_reconstructed,RMSE_reported,MAE_reported,RMSE_diff,MAE_diff
1,Decision tree,0.973625,0.808678,0.9715,0.8280,0.002125,-0.019322
3,Historical mean,1.022416,0.805497,1.0224,0.8055,0.000016,-0.000003
4,Linear regression: FIFA points,1.248072,1.019342,1.2481,1.0193,-0.000028,0.000042
0,Linear regression: FIFA rank,0.900249,0.721198,0.9002,0.7212,0.000049,-0.000002
2,Random forest,0.999938,0.815185,0.9767,0.7980,0.023238,0.017185


In [11]:
# The one fitted equation preserved from the original run (FIFA-rank linear regression) --
# recorded here as text since the fitted model object itself wasn't recoverable, only this equation.
reported_intercept = 0.2425
reported_rank_coef = -0.0181
print(f"Reported fitted equation: Y_hat = {reported_intercept:.4f} + ({reported_rank_coef:.4f}) * FIFA_rank")
print("A 10-position worsening predicts", 10 * reported_rank_coef, "fewer net goals per match.")

Reported fitted equation: Y_hat = 0.2425 + (-0.0181) * FIFA_rank
A 10-position worsening predicts -0.18100000000000002 fewer net goals per match.


## Extension: scoring the same 2018-fitted models against real 2026 World Cup results

Everything above reproduces the original baseline design (train on 2018, test on 2022) closely enough to trust the pipeline. Now we go further than the original notebook did: the 2026 World Cup has concluded, so `wc_2026_model_input.csv` (built alongside this notebook) gives us real observed outcomes for all 48 participating teams, plus FIFA rank/points sourced from the official 11 June 2026 pre-tournament ranking.

We do **not** refit anything here -- `fitted_models` (the linear regressions, decision tree, and random forest) and the historical-mean value were fit once, on 2018 only, above. We simply run them forward onto a second, independent test set they have never seen. This is the same "does it generalize beyond the data it was built on" question the `FIFA_Ranking_and_T5League_Modeling` notebook's 2026 evaluation asks of the linear-regression-on-roster-composition models -- here applied to the historical-mean/tree/forest family instead, so the project has one consistent answer across *every* model family the team tried, not just the linear ones.

In [12]:
wc2026_raw = pd.read_csv(WC2026_PATH)

# wc_2026_model_input.csv already has fifa_rank/fifa_ranking_points/mean_scoring_margin
# as its own columns -- no join against the FIFA rankings file needed here (unlike 2018/2022
# above), since that merge was already done when this CSV was built. Just rename to the
# schema run_baseline_models' fitted models expect: team_clean, fifa_rank, fifa_points,
# mean_scoring_margin.
test_2026 = wc2026_raw.rename(columns={
    "team": "team_clean",
    "fifa_ranking_points": "fifa_points",
})[["team_clean", "fifa_rank", "fifa_points", "mean_scoring_margin"]]

print(f"test (2026): {len(test_2026)} teams")
test_2026.head()

test (2026): 48 teams


,team_clean,fifa_rank,fifa_points,mean_scoring_margin
0,Spain,2.0,1874.71,1.625
1,Germany,10.0,1735.77,1.500
2,Netherlands,8.0,1753.57,1.500
3,Mexico,14.0,1687.48,1.400
4,Argentina,1.0,1877.27,1.375


In [13]:
y_2026 = test_2026["mean_scoring_margin"].values  # the real, observed outcome to score predictions against

results_2026 = []

# Historical mean: reuse the 2018 training average (NOT the 2026 average) -- this
# baseline, like every model below, only ever uses information from the original 2018 fit.
hist_pred_2026 = np.repeat(train["mean_scoring_margin"].mean(), len(test_2026))
results_2026.append({"Model": "Historical mean", **evaluate_regression(y_2026, hist_pred_2026)})

# fitted_models["rank_model"] etc. were fit on 2018 only, above -- .predict() here just
# runs that already-fitted line forward onto new (2026) inputs; nothing is refit.
rank_pred_2026 = fitted_models["rank_model"].predict(test_2026[["fifa_rank"]])
results_2026.append({"Model": "Linear regression: FIFA rank", **evaluate_regression(y_2026, rank_pred_2026)})

points_pred_2026 = fitted_models["points_model"].predict(test_2026[["fifa_points"]])
results_2026.append({"Model": "Linear regression: FIFA points", **evaluate_regression(y_2026, points_pred_2026)})

tree_pred_2026 = fitted_models["tree"].predict(test_2026[["fifa_rank", "fifa_points"]])
results_2026.append({"Model": "Decision tree", **evaluate_regression(y_2026, tree_pred_2026)})

forest_pred_2026 = fitted_models["forest"].predict(test_2026[["fifa_rank", "fifa_points"]])
results_2026.append({"Model": "Random forest", **evaluate_regression(y_2026, forest_pred_2026)})

results_2026_df = pd.DataFrame(results_2026).sort_values("RMSE").reset_index(drop=True)
results_2026_df

,Model,RMSE,MAE
0,Linear regression: FIFA rank,1.048746,0.849395
1,Decision tree,1.098777,0.794172
2,Random forest,1.248885,0.948995
3,Historical mean,1.335112,1.054805
4,Linear regression: FIFA points,1.497443,1.097846


### Side-by-side: 2022 test vs. 2026 test, same 2018-fitted models

In [14]:
# Join the 2022-test and 2026-test scores for each model side by side, so we can see
# directly whether each model's ranking (best to worst) holds up across both unseen tournaments,
# or whether a model that looked good on 2022 falls apart on 2026 (a sign it got lucky, not good).
combined = results_2022.merge(
    results_2026_df, on="Model", suffixes=("_2022", "_2026")
).sort_values("RMSE_2026").reset_index(drop=True)
combined

,Model,RMSE_2022,MAE_2022,RMSE_2026,MAE_2026
0,Linear regression: FIFA rank,0.900249,0.721198,1.048746,0.849395
1,Decision tree,0.973625,0.808678,1.098777,0.794172
2,Random forest,0.999938,0.815185,1.248885,0.948995
3,Historical mean,1.022416,0.805497,1.335112,1.054805
4,Linear regression: FIFA points,1.248072,1.019342,1.497443,1.097846


### Relating this back to the project

- **The FIFA-rank linear regression is still the strongest of this model family, out of sample, twice over.** It was the best 2022-test model in the original baseline and remains the best 2026-test model too -- the same fitted line (`Y_hat = 0.2425 - 0.0181 * FIFA_rank`) generalizes across two different tournaments it never saw during training. That is a stronger generalization claim than the 2022-only result could support on its own.
- **Decision tree and random forest do not clearly beat the simple linear model on either test set.** Both were tuned shallow (`max_depth=3`) specifically to avoid overfitting a 32-row training set, and that caution shows: they're competitive with linear regression but don't surpass it out-of-sample, reinforcing the same "simpler model, not more flexible model" conclusion the roster-composition modeling notebook reached independently with a completely different feature set.
- **FIFA Ranking Points (not rank) is the weakest predictor in both tests here too** -- consistent with the roster-composition notebook's finding that ordinal rank and roster-based features outperform raw ranking points as predictors of scoring margin.
- Together with the `FIFA_Ranking_and_T5League_Modeling` 2026 evaluation, this means **every model family in the project** -- historical mean, linear regression (rank and points), decision tree, random forest, and the roster-composition linear/neural-network models -- has now been checked against the real, concluded 2026 World Cup, not just an in-sample or single earlier holdout. That is the complete evidence base the final report can draw its model recommendation from.